   ... conceps in robotics called POSITION CONTROL ... (slightly different from
   forward kinematics... which is the math the evaluator uses to figure out
   where the TCP is in 3D space based on those angles...).

THE "SMART" MOTOR (Servos)
   ... the difference between `<general>` and `<position>` tags in the XML...

   The motors on this robot arm are set up as POSITION ACTUATORS (like standard
   hobby seros, but industrial grade)... do not have to do complex calculus
   to figure out how much electrical current (torque) needed to fight 
   gravity and lift the arm

   ... Instead, the motors have their own tiny, built-in computers running what's
   called a PD (Proportional-Derivative) Controller...

   1. You send the motor a COMMAND ANGLE (e.g. "Go to 1.5 radians")...
   2. The motor looks at where it currently is...
   3. The motor's internal computer calculates the exact amount of raw 
      electrical force needed to close that gap and applies it automatically...

THE TRAJECTORY "Breadcrumb Trial"
   If you just sent the final destination angle ... to the motor on frame 1, the
   motor would instntly blast maximum power to get there as fast as possible,
   violently jerking the arm.

   That is why `command_qpos` updates every single millisecond... Your 
   `trajectory.c` math is slicing that 90-deg journey into hundreds of tiny, 
   micro-angle waypoints (breadcrumbs) using that smooth S-curve.

   ... 

---

2. THE STATE MACHINE ENGINE
   These functions handle the transitions between the different phases of
   movement.
   - `enter_state` (The Gear Shifter): This function handles the transition from
     one movement phase to the next. When called, it records the current clock
     time (so `update` knows when the phase started), reads where the robot's
     joints physically are right now (`policy->start_qpos`), and calculates the 
     absolute destination for this new phase (`target_qpos = home_qpos + offset`)
   - `next_state`: A simple chronological roadmap. If the...
   - `state_duration`: A lookup table defining how many seconds each movement 
     phase should take. For example, the robot takes a leisurely 2.0 seconds to
     approach the rack... but a quicker 1.0 second to align itself.
   - `state_offset`: A lookup table that returns the correct array of 7 joint
     angles (your "tweaked by eye" waypoints) depending on the current state.

3. THE SAFETY && HARDWARE HELPERS
   These ensure the C code talks to the MuJoCo physics engine safely.
   - `read_current_qpos`: Loops through our 7 specific joint names, asks MuJoCo
     for their actual physical angles at this exact moment, and writes them into
     an array.
   - `clamp_actuator_ctrl`: A critical safety guard.. In your XML file, you 
     deined the torque or position limits for your motors... This function
     checks those XML limits... If your math accidentally asks a motor to move
     to a physically impossible angle, this functon forcefully clamps the value 
     to the maximum safe limit before sending the command to MuJoCo.    


---

   The  block of code is a classic CLI parser... Whenever ou run your compiled
   benchmark execuable from the terminal (for example, typing ...), the C program
   receives those typed wo.... The `for` loop acts as a scanner, moving through
   that array one word at a time to find specific configuration flags. This 
   allows you to dynamically change which XML scene you are loading or how many
   times the simulation runs without having to constantly edit your source code
   and recompile the project...

   Mechanically, it uses `strcmp` ... to hunt for known flags like `--scene` or
   `--trials`... The most critical part of this loop is the `i + 1 >= argc` 
   safety check.... if a use forget 